In [4]:
!pip install sympy --quiet
!pip install pycryptodome --quiet

from sympy import mod_inverse
import random
from typing import List, Tuple

def make_shares(secret: int, n: int) -> List[int]:
    MOD = 10**9+7
    shares = [random.randrange(0, MOD) for _ in range(n-1)]
    final = (secret - sum(shares)) % MOD
    shares.append(final)
    return shares

def distribute_and_compute_sum(values: List[int], parties: List[str]) -> Tuple[int, dict]:
    n = len(parties)
    MOD = 10**9+7
    sent = {p: {} for p in parties}
    for i, secret in enumerate(values):
        shares = make_shares(secret, n)
        for j, receiver in enumerate(parties):
            if receiver not in sent[parties[i]]:
                sent[parties[i]][receiver] = []
            sent[parties[i]][receiver].append(shares[j])
    local_sums = {}
    for receiver in parties:
        s = 0
        for sender in parties:
            s = (s + sent[sender][receiver][0]) % MOD
        local_sums[receiver] = s
    global_sum = sum(local_sums.values()) % MOD
    return global_sum, local_sums

parties = ["Ahmed", "Ali", "Rashida"]
values = [100, 70, 50]
global_sum, local_sums = distribute_and_compute_sum(values, parties)
print("Part I — MPC Sum (additive secret sharing)")
print("Private values:", dict(zip(parties, values)))
print("Local sums (each party):", local_sums)
print("Reconstructed global sum:", global_sum)
print()

PRIME = 2**127 - 1

def eval_poly(coeffs, x, p=PRIME):
    res = 0
    for a in reversed(coeffs):
        res = (res * x + a) % p
    return res

def shamir_split(secret: int, k: int, n: int, p=PRIME):
    if k > n:
        raise ValueError("k must be <= n")
    coeffs = [secret] + [random.randrange(0, p) for _ in range(k-1)]
    shares = []
    for i in range(1, n+1):
        x = i
        y = eval_poly(coeffs, x, p)
        shares.append((x, y))
    return shares

def lagrange_interpolate(x_samples: List[int], y_samples: List[int], p=PRIME) -> int:
    k = len(x_samples)
    secret = 0
    for i in range(k):
        xi, yi = x_samples[i], y_samples[i]
        num, den = 1, 1
        for j in range(k):
            if i == j:
                continue
            xj = x_samples[j]
            num = (num * (-xj)) % p
            den = (den * (xi - xj)) % p
        inv_den = mod_inverse(den, p)
        lag = num * inv_den % p
        secret = (secret + yi * lag) % p
    return secret

secret_value = 12345
k, n = 3, 5
shares = shamir_split(secret_value, k, n)
print("Part II — Shamir's Secret Sharing (k={}, n={})".format(k, n))
print("Original secret:", secret_value)
print("Shares (x, y) sample:", shares)

chosen = random.sample(shares, k)
xs = [x for x, y in chosen]
ys = [y for x, y in chosen]
reconstructed = lagrange_interpolate(xs, ys)
print("Reconstructed from chosen shares:", reconstructed)

try:
    chosen2 = random.sample(shares, k-1)
    xs2 = [x for x, y in chosen2]
    ys2 = [y for x, y in chosen2]
    rec2 = lagrange_interpolate(xs2, ys2)
    print("Reconstructed with k-1 shares (incorrect):", rec2)
except Exception as e:
    print("Cannot reconstruct with fewer than k shares:", e)
print()

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import hashlib

def rand_label():
    return get_random_bytes(16)

def encrypt_label(key: bytes, label: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_ECB)
    return cipher.encrypt(label)

def decrypt_label(key: bytes, ciphertext: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_ECB)
    return cipher.decrypt(ciphertext)

def garble_and(a_bit: int, b_bit: int):
    A0, A1 = rand_label(), rand_label()
    B0, B1 = rand_label(), rand_label()
    OUT0, OUT1 = rand_label(), rand_label()
    def key_of(l1, l2):
        return hashlib.sha256(l1 + l2).digest()[:16]
    table = {}
    inputs = [(0,0),(0,1),(1,0),(1,1)]
    for ia, ib in inputs:
        in_label_a = A1 if ia==1 else A0
        in_label_b = B1 if ib==1 else B0
        out_label = OUT1 if (ia & ib)==1 else OUT0
        key = key_of(in_label_a, in_label_b)
        ctxt = encrypt_label(key, out_label)
        table[(ia,ib)] = ctxt
    eval_label_a = A1 if a_bit==1 else A0
    eval_label_b = B1 if b_bit==1 else B0
    key_eval = key_of(eval_label_a, eval_label_b)
    outlabel = decrypt_label(key_eval, table[(a_bit,b_bit)])
    out_bit = 1 if outlabel == OUT1 else 0
    return out_bit

print("Part III — Garbled Circuit (AND gate) example")
res = garble_and(1, 0)
print("Inputs: A=1, B=0")
print("Computed A AND B =", res)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.4 MB/s eta 0:00:00
Part I — MPC Sum (additive secret sharing)
Private values: {'Ahmed': 100, 'Ali': 70, 'Rashida': 50}
Local sums (each party): {'Ahmed': 172925521, 'Ali': 87146130, 'Rashida': 739928576}
Reconstructed global sum: 220

Part II — Shamir's Secret Sharing (k=3, n=5)
Original secret: 12345
Shares (x, y) sample: [(1, 156408529144762468180485873138866164124), (2, 135338164213462715272297119354028996185), (3, 106930088666569973007121042361372614255), (4, 71184302504084241384957642160897018334), (5, 28100805726005520405806918752602208422)]
Reconstructed from chosen shares: 12345
Reconstructed with k-1 shares (incorrect): 22013131846778968070961969623457654318

Part III — Garbled Circuit (AND gate) example
Inputs: A=1, B=0
Computed A AND B = 0
